Zadania 1-5 wykonac za pomocą telneta

**Uwaga** W poniższych zadaniach zakładamy, iż serwer powinien obsługiwać tylko jednego klienta w danej chwili.

In [ ]:
import base64
import os
import socket
import threading

Funkcje i dane wspólne

In [ ]:
HOST = "127.0.0.1"
PORT = 110
USER = "pasinf2017@infumcs.edu"
PASS = "P4SInf2017"

In [ ]:
def connect():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect((HOST, PORT))
    return s


def recv_line(s):
    data = b""
    while not data.endswith(b"\r\n"):
        data += s.recv(1)
    return data.decode("ascii", errors="replace").strip()


def recv_multiline(s):
    lines = []
    while True:
        line = recv_line(s)
        if line == ".":
            break
        if line.startswith(".."):
            line = line[1:]
        lines.append(line)
    return lines


def send_cmd(s, cmd):
    s.sendall((cmd + "\r\n").encode("ascii"))


def login(s):
    recv_line(s)
    send_cmd(s, f"USER {USER}")
    recv_line(s)
    send_cmd(s, f"PASS {PASS}")
    resp = recv_line(s)
    if not resp.startswith("+OK"):
        raise RuntimeError(f"Login failed: {resp}")


def quit_session(s):
    send_cmd(s, "QUIT")
    recv_line(s)
    s.close()


def stat(s):
    send_cmd(s, "STAT")
    resp = recv_line(s)
    parts = resp.split()
    return int(parts[1]), int(parts[2])


def list_messages(s):
    send_cmd(s, "LIST")
    recv_line(s)
    lines = recv_multiline(s)
    result = []
    for line in lines:
        num, size = line.split()
        result.append((int(num), int(size)))
    return result


def retr(s, num):
    send_cmd(s, f"RETR {num}")
    recv_line(s)
    return recv_multiline(s)


def dele(s, num):
    send_cmd(s, f"DELE {num}")
    return recv_line(s)

1. Wykorzystując protokół telnet oraz wybrany serwer POP3, sprawdź, ile wiadomości znajduje się w skrzynce.
```
telnet 127.0.0.1 110
```
```
+OK POP3 server ready
USER pasinf2017@infumcs.edu
+OK pasinf2017@infumcs.edu welcome
PASS P4SInf2017
+OK Mailbox locked and ready
STAT
+OK 3 1325
QUIT
+OK Bye
```

2. Wykorzystując protokół telnet oraz wybrany serwer POP3, sprawdź, ile bajtów (w sumie) zajmują wiadomości znajdujące się w skrzynce.
```
telnet 127.0.0.1 110
```
```
+OK POP3 server ready
USER pasinf2017@infumcs.edu
+OK pasinf2017@infumcs.edu welcome
PASS P4SInf2017
+OK Mailbox locked and ready
STAT
+OK 3 1325
QUIT
+OK Bye
```

3. Wykorzystując protokół telnet oraz wybrany serwer POP3, sprawdź, ile bajtów zajmuje każda wiadomość (z osobna) znajdująca się w skrzynce.
```
telnet 127.0.0.1 110
```
```
+OK POP3 server ready
USER pasinf2017@infumcs.edu
+OK pasinf2017@infumcs.edu welcome
PASS P4SInf2017
+OK Mailbox locked and ready
LIST
+OK 3 messages
1 312
2 198
3 815
.
QUIT
+OK Bye
```

4. Wykorzystując protokół telnet, oraz wybrany serwer POP3, wyświetl treść wiadomości o największym rozmiarze.
```
telnet 127.0.0.1 110
```
```
+OK POP3 server ready
USER pasinf2017@infumcs.edu
+OK pasinf2017@infumcs.edu welcome
PASS P4SInf2017
+OK Mailbox locked and ready
LIST
+OK 3 messages
1 312
2 198
3 815
.
RETR 3
+OK 815 octets
From: charlie@example.com
To: pasinf2017@infumcs.edu
Subject: Image attached
Date: Wed, 03 Jan 2024 12:00:00 +0000
MIME-Version: 1.0
Content-Type: multipart/mixed; boundary="----=_Part_42_attachment"

------=_Part_42_attachment
Content-Type: text/plain; charset=utf-8

Please find the image attached.

------=_Part_42_attachment
Content-Type: image/png; name="attachment.png"
Content-Transfer-Encoding: base64
Content-Disposition: attachment; filename="attachment.png"

iVBORw0KGgoAAAANSUhEUgAAAAoAAAAKCAIAAAACUFjqAAAAoElEQVR4nA3J0RQAQAhFwTTSSCON
NDrnUjyNNNLIZHd+x8xwI4w0ymhDxhhrnGHmuBNOOuW0I2ecdc5/Bx5EkEEFHSiYYIOL34knkWRS
SSdKJtnk8nfhRRRZVNGFiim2uPrdeBNNNtV0o2aaba5/CxchUpRoITFixen34EMMOdTQg4YZdrj5
vfgSSy619KJlll1ufx9+xJFHHX3omGOPOx7JKonl1xxBAwAAAABJRU5ErkJggg==

------=_Part_42_attachment--
.
QUIT
+OK Bye
```

5. Wykorzystując protokół telnet oraz wybrany serwer POP3, usuń wiadomość o najmniejszym rozmiarze.
```
telnet 127.0.0.1 110
```
```
+OK POP3 server ready
USER pasinf2017@infumcs.edu
+OK pasinf2017@infumcs.edu welcome
PASS P4SInf2017
+OK Mailbox locked and ready
LIST
+OK 3 messages
1 312
2 198
3 815
.
DELE 2
+OK Message 2 deleted
LIST
+OK 2 messages
1 312
3 815
.
QUIT
+OK Bye
```

6. Napisz program klienta, który połączy się z wybranym serwerem POP3, a następnie wyświetli informację o tym, ile wiadomości znajduje się w skrzynce.

In [ ]:
def ex06():
    s = connect()
    login(s)
    count, _ = stat(s)
    print(f"Messages in mailbox: {count}")
    quit_session(s)

In [ ]:
ex06()

7. Napisz program klienta, który połączy się z wybranym serwerem POP3, a następnie wyświetli informację o tym, ile bajtów (w sumie) zajmują wiadomości znajdujące się w skrzynce.

In [ ]:
def ex07():
    s = connect()
    login(s)
    _, total = stat(s)
    print(f"Total size of messages: {total} bytes")
    quit_session(s)

In [ ]:
ex07()

8. Napisz program klienta, który połączy się z wybranym serwerem POP3, a następnie wyświetli informację o tym, ile bajtów zajmuje każda wiadomość (z osobna) znajdująca się w skrzynce.

In [ ]:
def ex08():
    s = connect()
    login(s)
    messages = list_messages(s)
    print(f"{'Message':>10}  {'Size (bytes)':>14}")
    print("-" * 28)
    for num, size in messages:
        print(f"{num:>10}  {size:>14}")
    quit_session(s)

In [ ]:
ex08()

9. Napisz program klienta, który połączy się z wybranym serwerem POP3, a następnie wyświetl treść wiadomości o największym rozmiarze.

In [ ]:
def ex09():
    s = connect()
    login(s)
    messages = list_messages(s)
    largest_num, largest_size = max(messages, key=lambda m: m[1])
    print(f"Largest message: #{largest_num} ({largest_size} bytes)\n")
    print("=" * 60)
    lines = retr(s, largest_num)
    print("\n".join(lines))
    print("=" * 60)
    quit_session(s)

In [ ]:
ex09()

10. Napisz program klienta, który połączy się z wybranym serwerem POP3, a następnie wyświetli wszystkie wiadomości znajdujące się w skrzynce.

In [ ]:
def ex10():
    s = connect()
    login(s)
    messages = list_messages(s)
    print(f"Mailbox contains {len(messages)} message(s).\n")

    for num, size in messages:
        print(f"{'=' * 60}")
        print(f"  Message #{num}  ({size} bytes)")
        print(f"{'=' * 60}")
        lines = retr(s, num)
        print("\n".join(lines))
        print()

    quit_session(s)

In [ ]:
ex10()

11. Napisz program klienta, który połączy się z wybranym serwerem POP3, a następnie pobierze z serwera wiadomość z załącznikiem (obrazkiem) i zapisze obrazek na dysk. Nazwa obrazka musi zgadzać się z nazwą załącznika podaną w mailu. Pamiętaj, że do przesyłania załączników binarnych w poczcie elektronicznej wykorzystywane jest kodowanie **Base64**.

In [ ]:
def parse_attachment(lines):
    raw = "\r\n".join(lines)

    boundary = None
    for line in lines:
        if "boundary=" in line.lower():
            boundary = line.split("boundary=")[-1].strip().strip('"')
            break

    if boundary is None:
        print("No MIME boundary found - message has no attachment.")
        return

    parts = raw.split(f"--{boundary}")

    for part in parts:
        if "Content-Disposition: attachment" not in part:
            continue

        filename = None
        for line in part.splitlines():
            if "filename=" in line.lower():
                filename = line.split("filename=")[-1].strip().strip('"')
                break

        if filename is None:
            filename = "attachment.bin"

        header_end = part.find("\r\n\r\n")
        if header_end == -1:
            header_end = part.find("\n\n")
            b64_data = part[header_end + 2:].strip()
        else:
            b64_data = part[header_end + 4:].strip()

        b64_data = b64_data.replace("--", "").strip()

        try:
            decoded = base64.decodebytes(b64_data.encode("ascii"))
        except Exception as e:
            print(f"Base64 decode error: {e}")
            return

        output_path = os.path.join(os.getcwd(), filename)
        with open(output_path, "wb") as f:
            f.write(decoded)

        print(f"Attachment saved: {output_path} ({len(decoded)} bytes)")
        return

    print("No attachment found in message.")

In [ ]:
def ex11():
    s = connect()
    login(s)
    messages = list_messages(s)

    for num, size in messages:
        lines = retr(s, num)
        raw = "\n".join(lines)
        if "content-disposition: attachment" in raw.lower():
            print(f"Found attachment in message #{num}")
            parse_attachment(lines)
            break
    else:
        print("No messages with attachments found.")

    quit_session(s)

In [ ]:
ex11()

12. Napisz program serwera, który działając pod adresem 127.0.0.1 oraz na określonym porcie TCP, będzie serwerem poczty, obsługującym protokół POP3. Nie realizuj faktycznego pobierania e-maili, tylko zasymuluj jego działanie tak, żeby napisany wcześniej klient POP3 mógł pobrać wiadomości. Pamiętaj o obsłudze przypadku, gdy klient poda niezaimplementowaną przez serwer komendę.

Wykonane w pliku ```server_zad12.py```